In [0]:
%sql
USE CATALOG dbr_dev;
CREATE SCHEMA IF NOT EXISTS skybound_raw
MANAGED LOCATION 'abfss://aviation-data@sadlsdev.dfs.core.windows.net/raw/';

CREATE SCHEMA IF NOT EXISTS skybound_bronze
MANAGED LOCATION 'abfss://aviation-data@sadlsdev.dfs.core.windows.net/bronze/';

CREATE SCHEMA IF NOT EXISTS skybound_silver
MANAGED LOCATION 'abfss://aviation-data@sadlsdev.dfs.core.windows.net/silver/';

CREATE SCHEMA IF NOT EXISTS skybound_gold
MANAGED LOCATION 'abfss://aviation-data@sadlsdev.dfs.core.windows.net/gold/';

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbr_dev.skybound_bronze.flights_bronze (
    icao24 STRING,
    callsign STRING,
    origin_country STRING,
    time_position STRING,
    last_contact STRING,
    longitude STRING,
    latitude STRING,
    baro_altitude STRING,
    on_ground STRING,
    velocity STRING,
    true_track STRING,
    vertical_rate STRING,
    sensors STRING,
    geo_altitude STRING,
    squawk STRING,
    spi STRING,
    position_source STRING,
    category STRING,
    ingestion_timestamp STRING 
)
USING DELTA
TBLPROPERTIES (delta.enableChangeDataFeed = true);

In [0]:
%sql
-- DROP TABLE IF EXISTS dbr_dev.skybound_bronze.weather_reports_bronze;

CREATE TABLE IF NOT EXISTS dbr_dev.skybound_bronze.weather_reports_bronze (
    icaoId STRING,
    receiptTime STRING, 
    obsTime STRING,
    reportTime STRING,
    temp STRING,
    dewp STRING,
    wdir STRING,
    wspd STRING,
    wgst STRING,
    visib STRING,
    altim STRING,
    wxString STRING,
    vertVis STRING,
    minT STRING,
    maxT STRING,
    qcField STRING, 
    metarType STRING,
    rawOb STRING,
    lat STRING,
    lon STRING,
    elev STRING,   
    name STRING,
    cover STRING,
    sky_cover STRING,
    cloud_base_ft STRING,
    fltCat STRING,
    _ingestion_timestamp STRING
)
USING DELTA
TBLPROPERTIES (delta.enableChangeDataFeed = true);


In [0]:
%sql
    
GRANT USE CATALOG ON CATALOG dbr_dev TO `aa152a97-9517-4f65-9892-a98bf829a554`;

GRANT ALL PRIVILEGES ON SCHEMA dbr_dev.skybound_raw TO `aa152a97-9517-4f65-9892-a98bf829a554`;

GRANT USE SCHEMA ON SCHEMA dbr_dev.skybound_bronze TO `aa152a97-9517-4f65-9892-a98bf829a554`;

GRANT SELECT, MODIFY ON TABLE dbr_dev.skybound_bronze.flights_bronze TO `aa152a97-9517-4f65-9892-a98bf829a554`;

SHOW GRANTS ON TABLE dbr_dev.skybound_bronze.flights_bronze;

GRANT SELECT, MODIFY ON TABLE dbr_dev.skybound_bronze.weather_reports_bronze TO `aa152a97-9517-4f65-9892-a98bf829a554`;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbr_dev.skybound_silver.weather_reports_silver (
    station_id STRING,
    station_name STRING,
    obs_time TIMESTAMP,
    receipt_time TIMESTAMP,
    temp_c  DOUBLE,
    dewpoint_c DOUBLE,
    wind_dir_degrees INT,
    wind_speed_kt INT,
    wind_gust_kt INT,
    visibility_sm DOUBLE,
    altim_hpa DOUBLE,
    wx_string STRING,
    cover STRING,
    sky_cover STRING,
    cloud_base_ft INT,
    flight_category STRING,
    metar_type STRING,
    latitude DOUBLE,
    longitude DOUBLE,
    elev_m INT,
    raw_ob STRING,
    _ingestion_timestamp STRING
  )
  USING DELTA
  TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS skybound_location
URL 'abfss://aviation-data@sadlsdev.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `dls_dev`);

In [0]:
%sql
ALTER TABLE
  dbr_dev.skybound_bronze.flights_bronze
SET TBLPROPERTIES
  (
    'delta.logRetentionDuration' = 'interval 1 hours',
    'delta.deletedFileRetentionDuration' = 'interval 1 hours'
  );